# TER Picopatt - Localisation

Importation des librairies principales et définition des dossiers.

Nos fonctions utilisées pour lire les données sont dans le fichier `functions.py`

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import radians, sin, cos, sqrt, atan2
import json
import geopandas as gpd
from shapely.geometry import shape, LineString, mapping
from ipywidgets import Output, VBox, HTML, HBox, Text, Button, Dropdown, Layout
from IPython.display import display
from ipyleaflet import Map, GeoJSON, GeomanDrawControl, LayersControl, basemaps, basemap_to_tiles
import re
import unicodedata
import functions as fc

# Dossiers de données et de sortie
DATA_NOZERO = Path("outputs/clean_nozeros")
FIG_FONT = Path("outputs/figures/fontaines")
OUT_LOCALISATION = Path("outputs/localisation")
fc.create_folder(OUT_LOCALISATION)
fc.create_folder(FIG_FONT)

# Affichage complet des colonnes
pd.set_option("display.max_columns", 200)

Lecture de tous les fichiers de données nettoyées (.csv, .xlsx) du dossier `clean_nozeros` dans un seul tableau, nettoyage des colonnes, puis vérification de la couverture temporelle et la répartition des mesures et ajout des colonnes `M_slot` (créneau horaire) et `date`.

In [11]:
bd = fc.load_all(DATA_NOZERO, False)

# Résumé
print("Chargement terminé")
print("Couverture :", bd['date'].min(), "->", bd['date'].max())
print("Parcours :", bd['track_id'].dropna().unique())
print("\nNombre de mesures par M_slot et parcours :")
print(
    bd.pivot_table(index="track_id", columns="M_slot", values="fichier_originaire", aggfunc="count")
       .fillna(0)
       .astype(int)
)

Chargement terminé
Couverture : 2024-10-29 -> 2025-01-16
Parcours : ['antigone' 'boulevards' 'ecusson']

Nombre de mesures par M_slot et parcours :
M_slot         M1     M2     M3     M4
track_id                              
antigone    29468  29924  31828  30237
boulevards  25603  34594  25785  29980
ecusson     30435  27041  26656  19730


# Localisation de fontaines

In [12]:
import geopandas as gpd
from shapely.geometry import shape, LineString, mapping
from ipywidgets import Output, VBox, HTML
from ipyleaflet import Map, GeoJSON, GeomanDrawControl, LayersControl, basemaps, basemap_to_tiles

# On travaille UNIQUEMENT avec les coordonnées on_track
REQUIRED_COLS = ["lon_ontrack", "lat_ontrack"]

missing = [c for c in REQUIRED_COLS if c not in bd.columns]
if missing:
    raise ValueError(
        f"Colonnes absentes dans bd : {missing}. "
    )

loc_df = bd.copy()

loc_df = loc_df[
    loc_df["lon_ontrack"].notna() &
    loc_df["lat_ontrack"].notna()
].copy()

if "date" in loc_df.columns:
    loc_df["date"] = pd.to_datetime(loc_df["date"], errors="coerce").dt.date

if "point_id" in loc_df.columns:
    loc_df["point_id"] = pd.to_numeric(loc_df["point_id"], errors="coerce")

if "track_id" in loc_df.columns:
    loc_df["track_id"] = loc_df["track_id"].astype(str).str.lower().str.strip()

gdf_all = gpd.GeoDataFrame(
    loc_df,
    geometry=gpd.points_from_xy(loc_df["lon_ontrack"], loc_df["lat_ontrack"]),
    crs="EPSG:4326"
)

print("Préparation terminée")
print("Nombre de points :", len(gdf_all))
print("Parcours disponibles :", sorted(gdf_all["track_id"].dropna().unique()))

Préparation terminée
Nombre de points : 341281
Parcours disponibles : ['antigone', 'boulevards', 'ecusson']


In [13]:
selected_reference_points_last = pd.DataFrame(
    columns=["track_id", "ref_idx", "lon_ontrack", "lat_ontrack"]
)
selected_reference_ranges_last = pd.DataFrame(
    columns=["track_id", "idx_start", "idx_end", "n_points"]
)
selection_catalog_last = pd.DataFrame(
    columns=["label", "track_id", "idx_start", "idx_end", "n_points"]
)
selected_polygon_last = None
selected_csv_path_last = None
reference_routes_last = {}

In [14]:
def slugify_name(text):
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text or "selection"


def compress_consecutive_indices(values):
    values = sorted(set(int(v) for v in values))
    if not values:
        return []

    ranges = []
    start = values[0]
    prev = values[0]

    for v in values[1:]:
        if v == prev + 1:
            prev = v
        else:
            ranges.append((start, prev, prev - start + 1))
            start = v
            prev = v

    ranges.append((start, prev, prev - start + 1))
    return ranges


def merge_overlapping_ranges(df):
    """
    Fusionne les plages qui se chevauchent ou se touchent,
    separement par label et parcours.
    """
    if df.empty:
        return df.copy()

    merged_rows = []

    df = df.sort_values(["label", "track_id", "idx_start", "idx_end"]).copy()

    for (label, track_id), sub in df.groupby(["label", "track_id"], dropna=False):
        current_start = None
        current_end = None

        for _, row in sub.iterrows():
            s = int(row["idx_start"])
            e = int(row["idx_end"])

            if current_start is None:
                current_start = s
                current_end = e
            elif s <= current_end + 1:
                current_end = max(current_end, e)
            else:
                merged_rows.append({
                    "label": label,
                    "track_id": track_id,
                    "idx_start": current_start,
                    "idx_end": current_end,
                    "n_points": current_end - current_start + 1
                })
                current_start = s
                current_end = e

        if current_start is not None:
            merged_rows.append({
                "label": label,
                "track_id": track_id,
                "idx_start": current_start,
                "idx_end": current_end,
                "n_points": current_end - current_start + 1
            })

    return pd.DataFrame(merged_rows).sort_values(
        ["label", "track_id", "idx_start"]
    ).reset_index(drop=True)


TABLE_STYLE = """
<style>
.loc-panel {
    border: 1px solid #e2e8f0;
    border-radius: 14px;
    padding: 12px 14px;
    background: #ffffff;
    box-shadow: 0 1px 2px rgba(15, 23, 42, 0.04);
}
.loc-title {
    font-size: 15px;
    font-weight: 700;
    color: #0f172a;
}
.loc-note {
    margin-top: 6px;
    color: #475569;
    font-size: 13px;
    line-height: 1.45;
}
.loc-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
    font-size: 13px;
}
.loc-table th,
.loc-table td {
    padding: 7px 10px;
    border-bottom: 1px solid #e2e8f0;
    text-align: left;
}
.loc-table thead th {
    background: #f8fafc;
    color: #334155;
    font-weight: 600;
}
.loc-table tbody tr:hover {
    background: #f8fafc;
}
</style>
"""


def dataframe_to_html(df, max_rows=20):
    if df is None or df.empty:
        return ""
    return df.head(max_rows).to_html(index=False, border=0, classes="loc-table")


def render_panel(widget, title_html, df=None, max_rows=20, note_html=""):
    note_block = f"<div class='loc-note'>{note_html}</div>" if note_html else ""
    widget.value = (
        TABLE_STYLE
        + "<div class='loc-panel'>"
        + f"<div class='loc-title'>{title_html}</div>"
        + note_block
        + dataframe_to_html(df, max_rows=max_rows)
        + "</div>"
    )

In [15]:
def build_ontrack_zone_selector(
    gdf,
    track="ecusson",
    date_min=None,
    date_max=None,
    m_slots=None,
    display_step=1,
    output_dir=OUT_LOCALISATION
):
    """
    Outil de selection sur tracé.

    Fonctionnalites :
    - plusieurs parcours
    - plusieurs selections
    - plusieurs labels dans un meme fichier
    - possibilite d'ajouter plusieurs dessins sous un meme label
    - sauvegarde finale d'un CSV catalogue
    """

    global selected_reference_points_last
    global selected_reference_ranges_last
    global selection_catalog_last
    global selected_polygon_last
    global selected_csv_path_last
    global reference_routes_last

    data = gdf.copy()

    state = {
        "last_draw_signature": None,
        "last_add_signature": None,
        "last_save_signature": None,
    }

    def normalize_tracks(track_value):
        if track_value is None:
            return None
        if isinstance(track_value, (list, tuple, set, np.ndarray, pd.Series)):
            return [str(t).lower().strip() for t in track_value]
        return [str(track_value).lower().strip()]

    def df_signature(df):
        if df is None or df.empty:
            return None
        return tuple(map(tuple, df.astype(str).itertuples(index=False, name=None)))

    tracks = normalize_tracks(track)

    route_colors = {
        "ecusson": "#ef4444",
        "antigone": "#2563eb",
        "boulevards": "#16a34a"
    }

    if "track_id" in data.columns and tracks is not None:
        data = data[data["track_id"].isin(tracks)].copy()

    if "date" in data.columns:
        if date_min is not None:
            data = data[data["date"] >= pd.to_datetime(date_min).date()]
        if date_max is not None:
            data = data[data["date"] <= pd.to_datetime(date_max).date()]

    if m_slots is not None and "M_slot" in data.columns:
        wanted = {str(x).upper().strip() for x in m_slots}
        data = data[data["M_slot"].astype(str).str.upper().isin(wanted)].copy()

    if data.empty:
        raise ValueError("Aucune donnee apres filtrage.")

    sort_cols = [c for c in ["track_id", "date", "M_slot", "point_id"] if c in data.columns]
    if sort_cols:
        data = data.sort_values(sort_cols).copy()

    tracks_present = sorted(data["track_id"].dropna().unique()) if "track_id" in data.columns else ["parcours"]

    reference_routes = {}
    route_layers = []

    for trk in tracks_present:
        sub = data[data["track_id"] == trk].copy()

        ref_cols = [c for c in ["date", "M_slot"] if c in sub.columns]
        if ref_cols and not sub.empty:
            ref_keys = sub[ref_cols].drop_duplicates().iloc[0].to_dict()
            mask = pd.Series(True, index=sub.index)
            for c, v in ref_keys.items():
                mask &= sub[c] == v
            ref_pass = sub.loc[mask].copy()
        else:
            ref_pass = sub.copy()

        if "point_id" in ref_pass.columns:
            ref_pass = ref_pass.sort_values("point_id")

        ref_pass = (
            ref_pass[["lon_ontrack", "lat_ontrack"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )

        ref_pass["ref_idx"] = np.arange(len(ref_pass))
        reference_routes[trk] = ref_pass.copy()

        ref_display = ref_pass.iloc[::max(1, display_step)].copy()

        if len(ref_display) >= 2:
            route_line = LineString(list(zip(ref_display["lon_ontrack"], ref_display["lat_ontrack"])))
            route_geojson = {
                "type": "FeatureCollection",
                "features": [{
                    "type": "Feature",
                    "geometry": mapping(route_line),
                    "properties": {"track_id": trk}
                }]
            }

            layer = GeoJSON(
                data=route_geojson,
                style={
                    "color": route_colors.get(trk, "#f59e0b"),
                    "weight": 4,
                    "opacity": 0.9
                }
            )
            layer.name = trk.replace("_", " ").title()
            route_layers.append(layer)

    reference_routes_last = reference_routes

    center = [float(data["lat_ontrack"].median()), float(data["lon_ontrack"].median())]

    try:
        base = basemap_to_tiles(basemaps.CartoDB.Voyager)
        base.name = "Plan"
        base.max_zoom = 22
        base.max_native_zoom = max(20, getattr(base, "max_native_zoom", 20) or 20)
        m = Map(
            center=center,
            zoom=18,
            max_zoom=22,
            min_zoom=3,
            zoom_snap=0.25,
            zoom_delta=0.5,
            scroll_wheel_zoom=True,
            double_click_zoom=True,
            box_zoom=True,
            touch_zoom=True,
            dragging=True,
            layout=Layout(height="420px", width="100%"),
            layers=(base,)
        )
    except Exception:
        m = Map(
            center=center,
            zoom=18,
            max_zoom=22,
            min_zoom=3,
            zoom_snap=0.25,
            zoom_delta=0.5,
            scroll_wheel_zoom=True,
            double_click_zoom=True,
            box_zoom=True,
            touch_zoom=True,
            dragging=True,
            layout=Layout(height="420px", width="100%")
        )

    selected_layer = GeoJSON(
        data={"type": "FeatureCollection", "features": []},
        style={
            "color": "#0ea5e9",
            "weight": 2,
            "opacity": 0.9,
            "fillColor": "#38bdf8",
            "fillOpacity": 0.18
        },
        point_style={
            "radius": 6,
            "color": "#0284c7",
            "fillColor": "#38bdf8",
            "fillOpacity": 0.9,
            "weight": 1
        }
    )
    selected_layer.name = "Selection"

    for layer in route_layers:
        m.add(layer)

    m.add(selected_layer)
    m.add(LayersControl(position="topright"))

    draw = GeomanDrawControl()
    draw.polygon = {"pathOptions": {"color": "#0ea5e9", "fillColor": "#38bdf8"}}
    draw.rectangle = {"pathOptions": {"color": "#0ea5e9", "fillColor": "#38bdf8"}}
    draw.circle = {}
    draw.polyline = {}
    draw.marker = {}
    draw.circlemarker = {}
    m.add(draw)

    current_panel = HTML()
    catalog_panel = HTML()
    action_panel = HTML()

    label_box = Text(
        value="",
        placeholder="Nouveau label, ex. place_comedie",
        layout=Layout(width="280px")
    )

    existing_label_dropdown = Dropdown(
        options=[("Reutiliser un label existant", "")],
        value="",
        layout=Layout(width="240px")
    )

    add_button = Button(
        description="Ajouter au catalogue",
        button_style="info",
        disabled=True,
        layout=Layout(width="190px")
    )

    save_name_box = Text(
        value="catalogue_localisations",
        placeholder="Nom du fichier CSV",
        layout=Layout(width="260px")
    )

    save_button = Button(
        description="Exporter le CSV",
        button_style="success",
        disabled=True,
        layout=Layout(width="150px")
    )

    clear_catalog_button = Button(
        description="Reinitialiser",
        button_style="warning",
        disabled=False,
        layout=Layout(width="130px")
    )

    title = HTML(   )

    label_controls = VBox(
        [
            HTML("<div style='font-size:13px; font-weight:700; color:#0f172a;'>Annotation</div><div style='font-size:12px; color:#64748b; margin-top:2px;'>Creer un nouveau label ou completer un label existant.</div>"),
            HBox([label_box, existing_label_dropdown, add_button], layout=Layout(gap="10px", flex_flow="row wrap"))
        ],
        layout=Layout(border="1px solid #e2e8f0", border_radius="14px", padding="12px", width="100%")
    )

    export_controls = VBox(
        [
            HTML("<div style='font-size:13px; font-weight:700; color:#0f172a;'>Export</div><div style='font-size:12px; color:#64748b; margin-top:2px;'>Enregistre le catalogue courant dans un CSV.</div>"),
            HBox([save_name_box, save_button, clear_catalog_button], layout=Layout(gap="10px", flex_flow="row wrap"))
        ],
        layout=Layout(border="1px solid #e2e8f0", border_radius="14px", padding="12px", width="100%")
    )

    controls = VBox([label_controls, export_controls], layout=Layout(gap="10px", margin="10px 0"))

    def refresh_catalog_widgets(active_label=""):
        labels = sorted(selection_catalog_last["label"].dropna().unique().tolist()) if not selection_catalog_last.empty else []
        existing_label_dropdown.options = [("Reutiliser un label existant", "")] + [(label, label) for label in labels]
        if active_label and active_label in labels:
            existing_label_dropdown.value = active_label
        elif existing_label_dropdown.value not in labels:
            existing_label_dropdown.value = ""
        save_button.disabled = selection_catalog_last.empty

        if selection_catalog_last.empty:
            render_panel(catalog_panel, "Catalogue actuel", note_html="Catalogue vide.")
        else:
            catalog_summary = (
                selection_catalog_last.groupby(["label", "track_id"], dropna=False)["n_points"]
                .sum()
                .reset_index()
                .rename(columns={"n_points": "n_points_total"})
                .sort_values(["label", "track_id"])
            )
            render_panel(
                catalog_panel,
                "Catalogue actuel",
                catalog_summary,
                max_rows=50,
                note_html=f"{len(catalog_summary)} ligne(s) dans le catalogue courant."
            )

    def get_target_label():
        typed = slugify_name(label_box.value) if str(label_box.value).strip() else ""
        chosen = slugify_name(existing_label_dropdown.value) if str(existing_label_dropdown.value).strip() else ""
        if typed:
            return typed
        if chosen:
            return chosen
        return ""

    def handle_draw(self, action, geo_json):
        global selected_reference_points_last
        global selected_reference_ranges_last
        global selected_polygon_last

        if action not in {"create", "edit", "cut"}:
            return

        feats = geo_json if isinstance(geo_json, list) else [geo_json]
        feat = feats[-1]

        geom = shape(feat["geometry"])
        current_geom_sig = (action, geom.wkb_hex)

        if state["last_draw_signature"] == current_geom_sig:
            return
        state["last_draw_signature"] = current_geom_sig

        selected_polygon_last = geom

        selected_points_frames = []
        selected_ranges_rows = []
        preview_features = []

        for trk, ref_df in reference_routes.items():
            ref_gdf = gpd.GeoDataFrame(
                ref_df.copy(),
                geometry=gpd.points_from_xy(ref_df["lon_ontrack"], ref_df["lat_ontrack"]),
                crs="EPSG:4326"
            )

            mask = ref_gdf.geometry.intersects(geom)
            sel = ref_df.loc[mask, ["ref_idx", "lon_ontrack", "lat_ontrack"]].copy()

            if sel.empty:
                continue

            sel.insert(0, "track_id", trk)
            selected_points_frames.append(sel)

            for idx_start, idx_end, n_points in compress_consecutive_indices(sel["ref_idx"].tolist()):
                selected_ranges_rows.append({
                    "track_id": trk,
                    "idx_start": idx_start,
                    "idx_end": idx_end,
                    "n_points": n_points
                })

            ref_preview = ref_gdf.loc[mask].iloc[::max(1, display_step)].copy()
            if not ref_preview.empty:
                preview_features.extend(json.loads(ref_preview.to_json())["features"])

        if not selected_points_frames:
            selected_layer.data = {"type": "FeatureCollection", "features": []}
            selected_reference_points_last = pd.DataFrame(
                columns=["track_id", "ref_idx", "lon_ontrack", "lat_ontrack"]
            )
            selected_reference_ranges_last = pd.DataFrame(
                columns=["track_id", "idx_start", "idx_end", "n_points"]
            )
            add_button.disabled = True
            render_panel(current_panel, "Selection courante", note_html="Aucun point de reference trouve dans la zone selectionnee.")
            render_panel(action_panel, "Detail par parcours", note_html="Aucun parcours retenu.")
            return

        selected_reference_points_last = (
            pd.concat(selected_points_frames, ignore_index=True)
            .sort_values(["track_id", "ref_idx"])
            .reset_index(drop=True)
        )

        selected_reference_ranges_last = (
            pd.DataFrame(selected_ranges_rows)
            .sort_values(["track_id", "idx_start"])
            .reset_index(drop=True)
        )

        selected_layer.data = {
            "type": "FeatureCollection",
            "features": preview_features
        }

        add_button.disabled = False

        by_track = (
            selected_reference_points_last.groupby("track_id")
            .size()
            .rename("n_points_reference")
            .reset_index()
        )

        render_panel(
            current_panel,
            "Selection courante",
            selected_reference_ranges_last.copy(),
            max_rows=20,
            note_html=(
                f"Points de reference retenus : {len(selected_reference_points_last)} | "
                f"Plages compactes : {len(selected_reference_ranges_last)}"
            )
        )
        render_panel(
            action_panel,
            "Detail par parcours",
            by_track,
            max_rows=10,
            note_html=f"{len(by_track)} parcours touche(s)."
        )

    def handle_add(_):
        global selection_catalog_last

        if selected_reference_ranges_last.empty:
            render_panel(action_panel, "Mise a jour", note_html="Aucune selection courante a ajouter.")
            return

        label = get_target_label()
        if not label:
            render_panel(action_panel, "Mise a jour", note_html="Renseigne un nouveau label ou choisis-en un existant.")
            return

        add_sig = (label, df_signature(selected_reference_ranges_last))
        if state["last_add_signature"] == add_sig:
            return
        state["last_add_signature"] = add_sig

        to_add = selected_reference_ranges_last.copy()
        to_add.insert(0, "label", label)

        selection_catalog_last = pd.concat(
            [selection_catalog_last, to_add],
            ignore_index=True
        )

        selection_catalog_last = merge_overlapping_ranges(selection_catalog_last)

        label_box.value = ""
        refresh_catalog_widgets(active_label=label)
        render_panel(
            action_panel,
            "Mise a jour",
            note_html=f"<span style='color:#047857; font-weight:600;'>Selection ajoutee au label <b>{label}</b>.</span>"
        )

    def handle_save_catalog(_):
        global selected_csv_path_last

        if selection_catalog_last.empty:
            render_panel(action_panel, "Export", note_html="Le catalogue est vide.")
            return

        file_name = slugify_name(save_name_box.value)
        save_sig = (file_name, df_signature(selection_catalog_last))
        if state["last_save_signature"] == save_sig:
            return
        state["last_save_signature"] = save_sig

        csv_path = output_dir / f"{file_name}.csv"
        to_save = merge_overlapping_ranges(selection_catalog_last.copy())
        to_save.to_csv(csv_path, index=False)
        selected_csv_path_last = csv_path

        render_panel(
            action_panel,
            "Export",
            note_html=(
                f"<span style='color:#047857; font-weight:600;'>Catalogue CSV sauvegarde.</span>"
                f"<br>{csv_path}"
                f"<br><span style='color:#64748b;'>Colonnes : {', '.join(to_save.columns)}</span>"
            )
        )

    def handle_clear_catalog(_):
        global selection_catalog_last

        selection_catalog_last = pd.DataFrame(
            columns=["label", "track_id", "idx_start", "idx_end", "n_points"]
        )
        refresh_catalog_widgets()
        render_panel(action_panel, "Mise a jour", note_html="Catalogue reinitialise.")

    def handle_dropdown_change(change):
        if change["name"] == "value" and change["new"]:
            label_box.value = change["new"]

    existing_label_dropdown.observe(handle_dropdown_change, names="value")
    draw.on_draw(handle_draw)
    add_button.on_click(handle_add)
    save_button.on_click(handle_save_catalog)
    clear_catalog_button.on_click(handle_clear_catalog)

    render_panel(current_panel, "Selection courante", note_html="Trace une zone pour afficher la selection retenue.")
    render_panel(action_panel, "Detail par parcours", note_html="Aucune selection pour le moment.")
    refresh_catalog_widgets()

    return VBox([title, m, controls, current_panel, catalog_panel, action_panel], layout=Layout(gap="12px"))

In [16]:
ui = build_ontrack_zone_selector(
    gdf_all,
    track=["ecusson", "antigone", "boulevards"],
    date_min="2024-10-01",
    date_max="2025-01-31",
    m_slots=["M1", "M2", "M3", "M4"],
    display_step=1,
    output_dir=OUT_LOCALISATION
)

ui